<a href="https://colab.research.google.com/github/MashrufAhmed-470/machine-learning/blob/main/AI_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Baking Chatbot

Robert Heller and Mashruf Ahmed
<br>
<br>
References:

Tutorial Series by Patrick Loeber: https://www.youtube.com/watch?v=RpWeNzfSUHw&ab_channel=PatrickLoeber

Tutorial by Rishi Sidhu: https://medium.com/x8-the-ai-community/build-your-first-chatbot-in-python-334247814900

Telegram / Python Tutorial: https://www.youtube.com/watch?v=CNhswOLqeLM&ab_channel=Indently



In [ ]:
#!pip install python-telegram-bot
!pip install python-telegram-bot==13.7

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.1/490.1 KB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 KB 4.7 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 5.3.0
    Uninstalling cachetools-5.3.0:
      Successfully uninstalled cachetools-5.3.0


In [ ]:
token = '6180406712:AAEFPknU0d2nmHWv4zBMk_H2cmgTKLXaY8c'

In [ ]:
import string
import random
import json
from google.colab import files
from google.colab import drive

# Natural Language Toolkit
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import PunktSentenceTokenizer

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from telegram.ext import *

import pandas as pd

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('nps_chat')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package nps_chat to /root/nltk_data...
[nltk_data]   Unzipping corpora/nps_chat.zip.


True

In [ ]:
current_word = ""

remove_punctuation = dict((ord(punct), None) for punct in string.punctuation)

In [ ]:
def tokenize(words):
  return nltk.word_tokenize(words.lower().translate(remove_punctuation))

stemmer = PorterStemmer()

def bag_of_words(tokenized_words, all_words):
  tokenized_words = [stem(current_word) for current_word in tokenized_words]
  bag = np.zeros(len(all), dtype=np.float32)
  for idx, current_word in enumerate(all):
    if current_word in tokenized_words:
      bag[idx] = 1.0

  return bag

def stem(indiv_word):
  return stemmer.stem(indiv_word.lower())

In [ ]:
# Add intents2.json JSON file

myFile = files.upload()

Saving intents2.json to intents2.json


In [ ]:
# Model (Neural Network)
# (1) Help from https://stackoverflow.com/questions/43080583/attributeerror-cannot-assign-module-before-module-init-call


class NeuralNetwork(nn.Module):
  def __init__(self, input_size, hidden_size, num_classes):
    super(NeuralNetwork, self).__init__() # (1)
    self.l1 = nn.Linear(input_size, hidden_size)
    self.l2 = nn.Linear(hidden_size, hidden_size)
    self.l3 = nn.Linear(hidden_size, num_classes)
    self.relu = nn.ReLU()

  def forward(self, x):
    out = self.l1(x)
    out = self.relu(out)
    out = self.l2(out)
    out = self.relu(out)
    out = self.l3(out)
    return out

In [ ]:
# Training

with open('intents2.json', 'r') as f:
  json_data = json.load(f)

all = []
tags = []
xy = []

for intent in json_data['intents']:
  tag = intent['tag']
  tags.append(tag)
  for pattern in intent['patterns']:
    current_word = tokenize(pattern)
    all.extend(current_word)
    xy.append((current_word, tag))

ignore = ['?','!','.',',']
all = [stem(current_word) for current_word in all if current_word not in ignore]
all = sorted(set(all)) # Not necessary, but helps
tags = sorted(set(tags)) # Not necessary, but helps

X_train = []
y_train = []

for (pattern_words, tag) in xy:
  bag = bag_of_words(pattern_words, all)
  X_train.append(bag)

  label = tags.index(tag)
  y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

class chatbotDS(Dataset):
  def __init__(self):
    self.numSamples = len(X_train)
    self.x_data = X_train
    self.y_data = y_train

  #
  def __getitem__(self,index):
    return self.x_data[index], self.y_data[index]

  def __len__(self):
    return self.numSamples

# Hyperparameters
batch_size = 8
hidden_size = 8
output_size = len(tags)
input_size = len(all)
learning_rate = 0.001
num_epochs = 400 #300-400 is a good range

ds = chatbotDS()
train_loader = DataLoader(dataset=ds, batch_size=batch_size, shuffle=True,
                          num_workers = 2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = NeuralNetwork(input_size, hidden_size, output_size)

criteria = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
  for (wordz, labels) in train_loader:
    wordz = wordz.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(wordz)
    loss = criteria(outputs, labels)

    loss.backward()
    optimizer.step()

  if (epoch +1) % 100 == 0:
    print(f'epoch {epoch+1}/{num_epochs}, loss={loss.item():.4f}')

print(f'final loss, loss={loss.item():.4f}')

stored_data = {
    "model_state": model.state_dict(),
    "input_size": input_size,
    "output_size": output_size,
    "hidden_size": hidden_size,
    "all": all,
    "tags": tags
}

file = "data.pth"
torch.save(stored_data, file)

epoch 100/400, loss=0.6650
epoch 200/400, loss=0.0124
epoch 300/400, loss=0.0037
epoch 400/400, loss=0.0065
final loss, loss=0.0065


In [ ]:
# Chat

runbot = False # Set runbot to True if testing or using the chatbot through Google Colab
running = False

model.load_state_dict(stored_data["model_state"])
model.eval()
chatbotName = "Baking Bot"

# Code for running chatbot through Telegram

def start_command(update, context):
  update.message.reply_text('Hello, I am BakeBot. How can I help you?')

def handle_response(text: str) -> str:
  text = tokenize(text)
  X = bag_of_words(text, all)
  X = X.reshape(1, X.shape[0])
  X = torch.from_numpy(X)

  output = model(X)
  _, predicted = torch.max(output, dim=1) # dim could be 0
  tag = tags[predicted.item()]

  probability = torch.softmax(output, dim=1)
  prob = probability[0][predicted.item()]

  if (prob.item() > 0.60):
    for intent in json_data["intents"]:
      if tag == intent["tag"]:
        return f"{random.choice(intent['responses'])}"
  else:
    return "Sorry, I do not understand your request. Please rephrase."

def handle_message(update,context):
  message_type = update.message.chat.type
  text = str(update.message.text).lower()
  response = ''

  print(f'User ({update.message.chat.id}) says: "{text}" in {message_type}')

  if message_type != 'group':
    if '@bakebot_bot' in text:
      new_text = text.replace('@bakebot_bot', '').strip()
      response = handle_response(new_text)
    else:
      response = handle_response(text)

    update.message.reply_text(response)

def error(update, context):
  print(f'Update {update} caused error: {context.error}')

if __name__ == '__main__':
  updater = Updater(token, use_context=True)
  dp = updater.dispatcher

  dp.add_handler(CommandHandler('start', start_command))
  dp.add_handler(MessageHandler(Filters.text, handle_message))
  dp.add_error_handler(error)

  updater.start_polling(1.0)
  updater.idle()

# Code for running chatbot through Google Colab

while (runbot == True):
  print("BAKEBOT")
  print("")

  userName = input("Please enter your name. ")
  print("")

  print(f"Hello {userName}, welcome to BakeBot.")
  print("Please type \"start\" or \"hello\" to begin. Or, type \"exit\" to end the program.")
  print("")

  userInput = input("" + userName + ": ")
  userInput = userInput.lower()
  print("")

  if ((userInput == 'start') or (userInput == 'hello')):
    print(f"{chatbotName}: Hello {userName}, what can I do for you?")
    print(f"{chatbotName}: Here is a list of what I can perform:")
    print("- Determine the best meal depending on your ingredients.")
    print("- Show the directions for preparing a meal.")
    print("- List the required ingredients for a particular meal.")
    print("- Something..")
    print("")

    running = True
  elif ((userInput == 'exit')):
    print(f"{chatbotName}: Thanks {userName}, have a nice day.")
    runbot = False
    running = False
    break
  else:
    print("Please try again.")
    continue

  while (running == True):
      #Chatbot AI runs here

      userInput2 = input("" + userName + ": ")

      if (userInput2 == 'exit'):
        print(f"{chatbotName}: Have a nice day, {userName}!")
        runbot = False;
        running = False;
        break

      userInput2 = tokenize(userInput2)
      X = bag_of_words(userInput2, all)
      X = X.reshape(1, X.shape[0])
      X = torch.from_numpy(X)

      output = model(X)
      _, predicted = torch.max(output, dim=1) # dim could be 0
      tag = tags[predicted.item()]

      probability = torch.softmax(output, dim=1)
      prob = probability[0][predicted.item()]

      if (prob.item() > 0.60):
        for intent in json_data["intents"]:
          if tag == intent["tag"]:
            print(f"{chatbotName}: {random.choice(intent['responses'])}")
            print("")
      else:
        print(f"{chatbotName}: Sorry, I do not understand your request. Please rephrase.")
        print("")